# Brochure Generator

Generates a company brochure from its website, translates it, and serves both through a Gradio UI.
Uses the Gemini API throughout (via its OpenAI-compatible endpoint).

In [ ]:
import os
import json
from enum import Enum
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from scraper import fetch_website_links, fetch_website_contents

In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)
google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file!")
elif not google_api_key.startswith(("AIz", "AQ.")):
    print("An API key was found, but it doesn't start with AIz or AQ.")
else:
    print("API key found and looks good so far!")

gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
MODEL = "gemini-3.5-flash-lite"

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.

"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000]  # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
def translator_system_prompt(language):
    system_prompt = f"""
You are a professional translator. You will be given a company brochure
written in markdown. Translate it into {language}, preserving markdown
formatting (headers, bullets, bold) and tone. Do not add commentary.
"""
    return system_prompt

In [ ]:
def get_translator_user_prompt(brochure, company, language):
    user_prompt = f"""
Translate the following company brochure for {company} into {language}.
Preserve all markdown formatting exactly. Do not add any extra commentary,
notes, or explanations - output only the translated brochure.

Brochure:

{brochure}
"""
    return user_prompt

In [ ]:
class Model(str, Enum):
    CHATGPT = "ChatGPT"
    CLAUDE = "Claude"

In [ ]:
def stream_chatgpt(company_name, url):
    yield "Researching the company website..."
    user_prompt = get_brochure_user_prompt(company_name, url)
    stream = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        stream=True,
        temperature=0.7,
    )
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

In [ ]:
def stream_claude(company_name, url):
    yield "Researching the company website..."
    user_prompt = get_brochure_user_prompt(company_name, url)
    stream = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        stream=True,
        temperature=0.3,
    )
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

In [ ]:
STREAM_FUNCTIONS = {
    Model.CHATGPT: stream_chatgpt,
    Model.CLAUDE: stream_claude,
}

def stream_brochure(company_name, url, model):
    yield from STREAM_FUNCTIONS[Model(model)](company_name, url)

In [ ]:
def stream_translated_brochure(brochure, company_name, language):
    if not brochure or not brochure.strip():
        raise gr.Error("Please generate a brochure first, then translate it.")
    yield "Translating..."
    stream = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": translator_system_prompt(language)},
            {"role": "user", "content": get_translator_user_prompt(brochure, company_name, language)},
        ],
        stream=True,
    )
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

In [ ]:
LANGUAGES = ["French", "Spanish", "German", "Japanese", "Arabic"]

with gr.Blocks(title="Company Brochure Generator") as view:
    gr.Markdown("# Company Brochure Generator")
    with gr.Row():
        company_input = gr.Textbox(label="Company name:")
        url_input = gr.Textbox(label="Landing page URL including http:// or https://")
        model_selector = gr.Dropdown([m.value for m in Model], label="Select model", value=Model.CHATGPT.value)
    generate_button = gr.Button("Generate Brochure")
    brochure_output = gr.Markdown(label="Brochure:")

    with gr.Row():
        language_selector = gr.Dropdown(LANGUAGES, label="Translate to", value=LANGUAGES[0])
        translate_button = gr.Button("Translate")
    translated_output = gr.Markdown(label="Translated Brochure:")

    generate_button.click(
        stream_brochure,
        inputs=[company_input, url_input, model_selector],
        outputs=brochure_output,
    )
    translate_button.click(
        stream_translated_brochure,
        inputs=[brochure_output, company_input, language_selector],
        outputs=translated_output,
    )

view.launch()